# 91 — Nodal source waveform and spectrum analysis

Catalog-driven replacement for `91_extract_empirical_source_waveforms_position_codes_*`.

This notebook uses nodal full-node shot gathers produced by `90_*` and metadata matches from the SQLite catalog. It no longer estimates source position from event files; it uses catalog source position and receiver geometry.

In [ ]:
from pathlib import Path
import sqlite3
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import read, Stream, Trace, UTCDateTime
from scipy.signal import detrend
from scipy.signal.windows import tukey

# Adjust if your repo lives somewhere else
REPO_ROOT = Path.cwd()
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "lib").exists():
        REPO_ROOT = parent
        break

import sys
LIB = REPO_ROOT / "lib"
if str(LIB) not in sys.path:
    sys.path.append(str(LIB))

try:
    from segy_tools.gather import plot_wiggle_gather_from_stream, plot_wiggle_gather_from_segy
except Exception as e:
    print("Could not import segy_tools plotting helpers:", e)

In [ ]:
# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------
PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")
CATALOG_DB = PROJECT_ROOT / "catalog" / "lbssp_shot_catalog.sqlite"
OUTPUT_ROOT = PROJECT_ROOT / "source_waveform_analysis"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

EXPORT_CSV = True
EXPORT_FIGURES = True

# Short window: closest approximation to early body-wave pulse.
SHORT_PRE_S = 0.005
SHORT_POST_S = 0.030

# Effective window: source + near-source ground coupling, but not full 0.5 s ground roll if avoidable.
EFFECTIVE_PRE_S = 0.010
EFFECTIVE_POST_S = 0.150

# Diagnostic window: includes ground roll/airwave/site response for inspection.
DIAGNOSTIC_PRE_S = 0.020
DIAGNOSTIC_POST_S = 0.500

WINDOWS = {
    "short_body_pulse": (SHORT_PRE_S, SHORT_POST_S),
    "effective_wavelet": (EFFECTIVE_PRE_S, EFFECTIVE_POST_S),
    "diagnostic_long": (DIAGNOSTIC_PRE_S, DIAGNOSTIC_POST_S),
}

NFFT = 8192
FMAX_PLOT = 500.0
TAPER_ALPHA = 0.5

assert CATALOG_DB.exists(), f"Catalog database not found: {CATALOG_DB}"
print(CATALOG_DB)

In [ ]:
def connect_catalog(db_path=CATALOG_DB):
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA busy_timeout = 30000")
    return conn


def read_table(conn, table):
    try:
        return pd.read_sql(f"SELECT * FROM {table}", conn)
    except Exception as e:
        print(f"Could not read {table}: {e}")
        return pd.DataFrame()


def parse_time(x):
    if pd.isna(x) or x in ("", None):
        return None
    try:
        return UTCDateTime(str(x))
    except Exception:
        try:
            return UTCDateTime(pd.to_datetime(x).to_pydatetime())
        except Exception:
            return None


def station_x_from_code(station):
    """Position-coded stations are cm along line: 08808 -> 88.08 m."""
    try:
        return float(str(station)) / 100.0
    except Exception:
        return np.nan


def load_stream_for_event(files_df, event_id, component=None, instrument_system=None):
    q = files_df[files_df["event_id"] == event_id].copy()
    if component is not None:
        q = q[q["component"].astype(str).str.upper() == component.upper()]
    if instrument_system is not None and "instrument_system" in q.columns:
        q = q[q["instrument_system"] == instrument_system]

    # Prefer 3C MiniSEED if no specific component requested.
    if component is None:
        q0 = q[(q["file_type"] == "mseed") | (q["component"].astype(str).str.upper() == "3C")]
        if len(q0):
            q = q0

    if len(q) == 0:
        raise FileNotFoundError(f"No files for event_id={event_id}, component={component}")

    # Read first available waveform file.
    row = q.iloc[0]
    f = Path(row["file_path"])
    if not f.exists():
        raise FileNotFoundError(f)
    return read(str(f))


def nearest_trace(st, source_x_m, component="Z", max_offset_m=None):
    """Return nearest Trace to source_x_m, optionally choosing a component suffix."""
    rows = []
    for i, tr in enumerate(st):
        cha = tr.stats.channel
        if component is not None and not cha.upper().endswith(component.upper()):
            continue
        x = station_x_from_code(tr.stats.station)
        if not np.isfinite(x):
            continue
        off = abs(x - source_x_m)
        if max_offset_m is not None and off > max_offset_m:
            continue
        rows.append((off, x, i, tr))
    if not rows:
        return None, None
    rows = sorted(rows, key=lambda r: r[0])
    return rows[0][3], {"offset_m": rows[0][0], "receiver_x_m": rows[0][1], "trace_index": rows[0][2]}


def get_pick_time_for_trace(picks_df, event_id, tr, preferred_phase="P"):
    if picks_df.empty:
        return None
    q = picks_df[picks_df["event_id"] == event_id].copy()
    for col, value in [("station", tr.stats.station), ("channel", tr.stats.channel)]:
        if col in q.columns:
            q = q[q[col].astype(str) == str(value)]
    if "phase" in q.columns and preferred_phase is not None:
        qp = q[q["phase"].astype(str).str.upper() == preferred_phase.upper()]
        if len(qp):
            q = qp
    for col in ["pick_time_utc", "pick_time", "consensus_time", "time"]:
        if col in q.columns and q[col].notna().any():
            return parse_time(q.iloc[0][col])
    return None


def extract_window_around_time(tr, center_time, pre_s, post_s, taper_alpha=TAPER_ALPHA):
    if center_time is None:
        raise ValueError("center_time is None")
    t1 = center_time - pre_s
    t2 = center_time + post_s
    pulse = tr.copy().trim(t1, t2, pad=True, fill_value=0)
    x = pulse.data.astype(float)
    if len(x) < 4:
        raise ValueError("window too short")
    x = detrend(x, type="linear")
    x = x - np.nanmean(x)
    w = tukey(len(x), alpha=taper_alpha)
    xt = x * w
    return pulse, x, xt


def amplitude_spectrum(x, dt, nfft=NFFT, normalize=True):
    freq = np.fft.rfftfreq(nfft, dt)
    amp = np.abs(np.fft.rfft(x, n=nfft))
    if normalize and np.nanmax(amp) > 0:
        amp = amp / np.nanmax(amp)
    return freq, amp


def plot_waveform_spectrum(x, xt, dt, title="", fmax=FMAX_PLOT):
    t = np.arange(len(x)) * dt
    freq, amp = amplitude_spectrum(xt, dt)
    fig, axes = plt.subplots(2, 1, figsize=(9, 6), constrained_layout=True)
    axes[0].plot(t, x, label="raw window")
    axes[0].plot(t, xt, label="tapered")
    axes[0].set_xlabel("Time since window start (s)")
    axes[0].set_ylabel("Amplitude")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].set_title(title)
    axes[1].plot(freq, amp)
    axes[1].set_xlim(0, fmax)
    axes[1].set_xlabel("Frequency (Hz)")
    axes[1].set_ylabel("Normalized amplitude")
    axes[1].grid(True, alpha=0.3)
    return fig, freq, amp


def event_label(row):
    parts = [str(row.get("line", "")), str(row.get("source_type", "")), str(row.get("source_x_m", "")), str(row.get("event_id", ""))]
    return " | ".join([p for p in parts if p and p != "nan"])

In [ ]:
conn = connect_catalog()
shot_events = read_table(conn, "shot_events")
shot_files = read_table(conn, "shot_gather_files")
trace_index = read_table(conn, "trace_index")
picks = read_table(conn, "picks")
receiver_geometry = read_table(conn, "receiver_geometry")

for name, df in [("shot_events", shot_events), ("shot_files", shot_files), ("trace_index", trace_index), ("picks", picks), ("receiver_geometry", receiver_geometry)]:
    print(name, df.shape)

## Select matched nodal events

In [ ]:
nodal = shot_events.copy()
if "instrument_system" in nodal.columns:
    nodal = nodal[nodal["instrument_system"].astype(str).str.lower().eq("nodal")].copy()

if "source_x_m" in nodal.columns:
    nodal["source_x_m"] = pd.to_numeric(nodal["source_x_m"], errors="coerce")

matched = nodal[nodal.get("source_x_m", pd.Series(index=nodal.index, dtype=float)).notna()].copy()
if "metadata_match_status" in matched.columns:
    print(matched["metadata_match_status"].value_counts(dropna=False))

print("nodal events", len(nodal), "matched/positioned", len(matched))
cols = [c for c in ["event_id", "line", "timewindow_label", "source_x_m", "source_type", "operator", "plate_type", "n_blows", "metadata_match_status", "matched_metadata_event_id"] if c in matched.columns]
display(matched[cols].head(30))

## Inspect nearest receiver source waveforms

In [ ]:
# Edit these filters as needed.
selected = matched.copy()
if "line" in selected.columns:
    selected = selected[selected["line"].astype(str).isin(["T1", "T1A", "T3", "T4"])]
selected = selected.head(30).copy()

figdir = OUTPUT_ROOT / "nodal_source_waveforms"
figdir.mkdir(parents=True, exist_ok=True)
rows = []

for _, ev in selected.iterrows():
    event_id = str(ev["event_id"])
    source_x_m = float(ev["source_x_m"])
    try:
        st = load_stream_for_event(shot_files, event_id, component=None, instrument_system="nodal")
    except Exception as e:
        print("Skipping", event_id, e)
        continue

    tr, ninfo = nearest_trace(st, source_x_m, component="Z")
    if tr is None:
        print("No nearest Z trace for", event_id)
        continue

    center = get_pick_time_for_trace(picks, event_id, tr)
    center_source = "pick"
    if center is None:
        # Fallback: use predicted travel time from shot_time/detection_time if available, else trace start.
        center = tr.stats.starttime
        center_source = "trace_start_placeholder"

    for window_name, (pre_s, post_s) in WINDOWS.items():
        try:
            pulse, x, xt = extract_window_around_time(tr, center, pre_s, post_s)
        except Exception as e:
            print("Window fail", event_id, window_name, e)
            continue
        freq, amp = amplitude_spectrum(xt, pulse.stats.delta)
        peak_freq = float(freq[np.nanargmax(amp)]) if len(freq) else np.nan
        rows.append({
            "event_id": event_id,
            "line": ev.get("line", None),
            "timewindow_label": ev.get("timewindow_label", None),
            "source_type": ev.get("source_type", None),
            "source_x_m": source_x_m,
            "operator": ev.get("operator", None),
            "plate_type": ev.get("plate_type", None),
            "n_blows": ev.get("n_blows", None),
            "metadata_match_status": ev.get("metadata_match_status", None),
            "matched_metadata_event_id": ev.get("matched_metadata_event_id", None),
            "station": tr.stats.station,
            "channel": tr.stats.channel,
            "receiver_x_m": ninfo["receiver_x_m"],
            "offset_m": ninfo["offset_m"],
            "window_name": window_name,
            "center_time_source": center_source,
            "window_length_s": pre_s + post_s,
            "peak_freq_hz_normalized_spectrum": peak_freq,
        })
        if EXPORT_FIGURES and window_name != "diagnostic_long":
            title = f"Nodal {event_id} {window_name}: xsrc={source_x_m:.2f}, xrec={ninfo['receiver_x_m']:.2f}, {tr.id}"
            fig, _, _ = plot_waveform_spectrum(x, xt, pulse.stats.delta, title=title)
            fig.savefig(figdir / f"{event_id}_{tr.stats.station}_{tr.stats.channel}_{window_name}.png", dpi=150)
            plt.close(fig)

summary = pd.DataFrame(rows)
display(summary.head())
if EXPORT_CSV:
    summary.to_csv(OUTPUT_ROOT / "nodal_source_waveform_summary.csv", index=False)
print("rows", len(summary))

## Repeated-shot groups for future stacking

In [ ]:
if len(matched):
    group_cols = [c for c in ["line", "timewindow_label", "source_x_m", "source_type", "operator"] if c in matched.columns]
    groups = matched.groupby(group_cols, dropna=False).size().reset_index(name="n_events").sort_values("n_events", ascending=False)
    display(groups.head(50))
    groups.to_csv(OUTPUT_ROOT / "nodal_repeated_shot_groups.csv", index=False)